In [1]:
!pip install -q openai

In [3]:
import json
import os
import re
import statistics
import subprocess
import threading
import time
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum, auto
from pathlib import Path
from typing import Callable, Optional
from openai import OpenAI

os.environ['OPENAI_API_KEY'] = ''
client = OpenAI()
MODEL = 'gpt-4o-mini'

In [22]:
# === Minimal agent loop (self-contained) ===
def run_agent_loop(
        messages,
        tools,
        tool_handlers,
        system='',
        max_iterations=20,
        on_tool_call=None,
        on_response=None
    ):

    # messages: list[dict] (N,) -- mutated in-place
    full = ([{'role':'system','content':system}] + messages) if system else list(messages)

    for _ in range(max_iterations):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=full,
            tools=tools if tools else None,
            tool_choice='auto' if tools else None,
        )

        if on_response:
            on_response(resp)

        msg = resp.choices[0].message
        reason = resp.choices[0].finish_reason
        ser = {'role':'assistant','content':msg.content}

        if msg.tool_calls:
            ser['tool_calls'] = [
                {
                    'id':tc.id,
                    'type':'function',
                    'function':{'name':tc.function.name,'arguments':tc.function.arguments}
                }
                for tc in msg.tool_calls
            ]

        full.append(ser)
        messages.append(ser)

        if reason != 'tool_calls':
            return messages

        for tc in msg.tool_calls:
            name = tc.function.name
            inp  = json.loads(tc.function.arguments)
            if on_tool_call:
                on_tool_call(name, inp)

            h = tool_handlers.get(name)
            try:
                result = h(**inp) if h else f'[ToolError] No handler: {name}'
            except Exception as exc:
                result = f'[ToolError] {type(exc).__name__}: {exc}'

            tm = {
                'role':'tool','tool_call_id':tc.id,
                'content':str(result)
            }
            full.append(tm)
            messages.append(tm)
    return messages

print(f'Client ready. Model: {MODEL}')

Client ready. Model: gpt-4o-mini


# 1) Observability

An agent in production is a black box. Without observability you cannot answer basic questions:
- Why did the agent take 12 seconds?
- Which tool was called 200 times?
- Why did this session cost $3?

**Observability = Logs + Metrics + Traces.**

- **Logs**: structured JSON records of every event (message received,
  tool called, LLM response received, error occurred).
- **Metrics**: aggregated numbers (tokens/turn, latency percentiles,
  error rates, cache hit ratio).
- **Traces**: causal chains linking a user message to every downstream
  action it triggered (span tree).

| Event | What to log |
|-------|-------------|
| LLM call start | model, prompt token count, session_id |
| LLM call end | latency_ms, output tokens, finish_reason |
| Tool call | tool_name, input summary, start_time |
| Tool result | output length, duration_ms, error? |
| Compaction | before_tokens, after_tokens, layer used |
| Session start/end | session_id, channel, turn count |

1. Each user request is a root **span**.
2. Every LLM call and tool call is a child span.
3. The trace is the full tree of spans.
4. This mirrors OpenTelemetry's data model

## 1.1 Event Log

In [23]:
@dataclass
class Span:
    # One unit of work in a trace.
    span_id: str
    parent_id: Optional[str]
    name: str
    start_ms: float
    end_ms: Optional[float] = None
    attrs: dict = field(default_factory=dict)
    error: Optional[str] = None

    @property
    def duration_ms(self):
        if self.end_ms is None: return None
        return round(self.end_ms - self.start_ms, 2)

    def finish(self, **attrs):
        self.end_ms = time.time() * 1000
        self.attrs.update(attrs)
        return self

    def to_dict(self):
        return {**self.__dict__, 'duration_ms': self.duration_ms}

class Tracer:
    '''
    Collects spans into traces. One Tracer per session.
    Thread-safe (multiple tool calls can be in flight simultaneously).
    '''

    def __init__(self, session_id):
        self.session_id = session_id
        # _spans: list[Span] (num_spans,)
        self._spans: list = []
        self._lock = threading.Lock()

    def start_span(self, name, parent_id=None, **attrs):
        # name: str -> Span (call .finish() when done)
        span = Span(
            span_id=uuid.uuid4().hex[:8],
            parent_id=parent_id,
            name=name,
            start_ms=time.time() * 1000,
            attrs=attrs,
        )
        with self._lock:
            self._spans.append(span)
        return span

    def record_error(self, span, error):
        span.error = str(error)
        span.finish()

    def summary(self):
        # -> str (human-readable trace summary)
        with self._lock:
            spans = list(self._spans)
        if not spans:
            return 'No spans recorded.'

        lines = [f'Trace [{self.session_id}]: {len(spans)} spans']

        for s in spans:
            indent = '  ' if s.parent_id else ''
            dur = f'{s.duration_ms:.0f}ms' if s.duration_ms else 'running'
            err = f' ERROR: {s.error}' if s.error else ''
            lines.append(f'{indent}  {s.name} [{dur}]{err}')
        return '\n'.join(lines)

class MetricsStore:
    '''
    Aggregates numerical metrics with rolling window support.
    Thread-safe.
    '''

    def __init__(self, window_size=100):
        # _data: dict[str, deque[float]] (num_metrics,)
        self._data: dict = defaultdict(lambda: deque(maxlen=window_size))
        self._lock = threading.Lock()

    def record(self, key, value):
        # key: str, value: float -> None
        with self._lock:
            self._data[key].append(value)

    def stats(self, key):
        # key: str -> dict (count, mean, p50, p95, p99, min, max)
        with self._lock:
            vals = list(self._data.get(key, []))

        if not vals:
            return {'count': 0}

        sorted_vals = sorted(vals)
        n = len(sorted_vals)

        def pct(p):
            return sorted_vals[min(int(n * p / 100), n-1)]

        return {
            'count': n,
            'mean': round(statistics.mean(vals), 2),
            'p50': round(pct(50), 2),
            'p95': round(pct(95), 2),
            'p99': round(pct(99), 2),
            'min': round(min(vals), 2),
            'max': round(max(vals), 2),
        }

    def report(self):
        # -> str (all metrics)
        with self._lock:
            keys = list(self._data.keys())
        lines = []
        for k in keys:
            s = self.stats(k)
            lines.append(f'  {k}: {s}')
        return 'Metrics:\n' + ('\n'.join(lines) or '  (empty)')


class AgentLogger:
    '''
    Structured JSON logger for agent events.
    Writes to both stdout (pretty) and an append-only JSONL file.
    '''

    def __init__(self, log_path='.logs/agent.jsonl'):
        self._path = Path(log_path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self.metrics = MetricsStore()

    def _write(self, event_type, **fields):
        record = {
            'ts': datetime.now().isoformat(),
            'event': event_type,
            **fields
        }

        with open(self._path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, default=str) + '\n')
        return record

    def llm_call(self, session_id, model, prompt_tokens):

        return self._write(
            'llm_call',
            session_id=session_id,
            model=model,
            prompt_tokens=prompt_tokens
        )

    def llm_response(
        self,
        session_id,
        latency_ms,
        output_tokens,
        finish_reason,
        cached=False
    ):

        self.metrics.record('llm_latency_ms', latency_ms)
        self.metrics.record('output_tokens', output_tokens)
        return self._write(
            'llm_response',
            session_id=session_id,
            latency_ms=latency_ms,
            output_tokens=output_tokens,
            finish_reason=finish_reason,
            cached=cached
        )

    def tool_call(self, session_id, tool_name, input_summary):

        self.metrics.record('tool_calls_total', 1)
        return self._write(
            'tool_call',
            session_id=session_id,
            tool=tool_name,
            input=input_summary
        )

    def tool_result(
        self,
        session_id,
        tool_name,
        duration_ms,
        output_len,
        error=None
    ):

        self.metrics.record(f'tool_{tool_name}_ms', duration_ms)

        return self._write(
            'tool_result',
            session_id=session_id,
            tool=tool_name,
            duration_ms=duration_ms,
            output_len=output_len,
            error=error
        )

    def error(self, session_id, message, exc=None):
        self.metrics.record('errors_total', 1)
        return self._write(
            'error',
            session_id=session_id,
            message=message,
            exc=str(exc) if exc else None
        )


# === Demo ===
logger  = AgentLogger()
tracer  = Tracer(session_id='demo_session')

# Simulate a turn
root = tracer.start_span('user_turn', user_message='What is 2+2?')
llm  = tracer.start_span('llm_call', parent_id=root.span_id, model=MODEL)
logger.llm_call('demo_session', MODEL, prompt_tokens=50)
llm.finish(output_tokens=20, finish_reason='stop')
logger.llm_response('demo_session', latency_ms=50, output_tokens=20, finish_reason='stop')
root.finish()

print(tracer.summary())
print()
print(logger.metrics.report())

Trace [demo_session]: 2 spans
  user_turn s]
    llm_call s]

Metrics:
  llm_latency_ms: {'count': 1, 'mean': 50, 'p50': 50, 'p95': 50, 'p99': 50, 'min': 50, 'max': 50}
  output_tokens: {'count': 1, 'mean': 20, 'p50': 20, 'p95': 20, 'p99': 20, 'min': 20, 'max': 20}


## 1.2 Cross-Session Search with FTS5

The `AgentLogger` logs events within a session.
Hermes adds **cross-session search**: index every conversation with
SQLite FTS5 (Full-Text Search) so the agent can answer questions like

> What did we decide about the auth refactor two weeks ago?

FTS5 is built into Python's stdlib `sqlite3` -- no extra dependencies.

**Why FTS5 and not a vector store?**

> FTS5 is exact keyword search (BM25 ranking). It is 10x simpler to deploy,
has zero cost, and works well for retrieving specific past decisions or
tool outputs by keyword. Vector search is better for semantic similarity;
FTS5 is better for factual recall ('what was the error message').

```python
store.index_session('sess_001', messages, summary='Fixed auth bug')
store.index_session('sess_002', messages, summary='Added rate limiting')

results = store.search('auth JWT')
# -> [{'session_id': 'sess_001', 'summary': 'Fixed auth bug',
#      'snippet': '...validate_token decodes the JWT...', 'rank': -1.4}]
```

In [24]:
import sqlite3
from pathlib import Path

class SessionSearchIndex:
    '''
    SQLite FTS5 index for cross-session full-text search.
    Indexes session summaries and message content.
    Enables the agent to recall past conversations by keyword.
    Modelled on Hermes FTS5 session search.
    '''

    def __init__(self, db_path='.sessions/search.db'):
        self._path = Path(db_path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._conn = sqlite3.connect(str(self._path), check_same_thread=False)
        self._setup()

    def _setup(self):
        # Create FTS5 virtual table if it does not exist.
        # FTS5 automatically builds an inverted index over all text columns.
        self._conn.executescript('''
            CREATE VIRTUAL TABLE IF NOT EXISTS sessions_fts USING fts5(
                session_id UNINDEXED,
                summary,
                content,
                created_at UNINDEXED
            );
        ''')
        self._conn.commit()

    def index_session(self, session_id, messages, summary=''):
        '''
        Index one session for search.

        session_id: str, messages: list[dict] (N,), summary: str -> None
        Concatenate all assistant + user text for the searchable content field
        messages: list[dict] (N,) -> content: str (all text, space-joined)
        '''

        content = ' '.join(
            m.get('content', '') or ''
            for m in messages
            if m.get('role') in ('user', 'assistant') and m.get('content')
        )

        # Remove any prior entry for this session (upsert via delete + insert)
        # To prevent multiple rows in database for same session_id
        self._conn.execute(
            'DELETE FROM sessions_fts WHERE session_id = ?', (session_id,)
        )

        # Re-insert the updated record
        self._conn.execute(
            'INSERT INTO sessions_fts(session_id, summary, content, created_at) VALUES (?,?,?,?)',
            (session_id, summary, content, datetime.now().isoformat()),
        )
        self._conn.commit()

    def search(self, query, top_k=5):
        '''
        Full-text search across all indexed sessions.
        FTS5 BM25 ranking: lower rank = more relevant.
        query: str -> list[dict] (top_k,) ranked by relevance
        '''

        rows = self._conn.execute('''
            SELECT session_id, summary, created_at,
                   snippet(sessions_fts, 2, '[', ']', '...', 20) AS snippet,
                   rank
            FROM sessions_fts
            WHERE sessions_fts MATCH ?
            ORDER BY rank
            LIMIT ?
        ''', (query, top_k)).fetchall()

        return [
            {'session_id': r[0],
             'summary': r[1],
             'created_at': r[2],
             'snippet': r[3],
             'rank': round(r[4], 3)}
            for r in rows
        ]

    def summarise_and_index(self, session_id, messages):
        '''
        Auto-generate a summary via LLM then index.
        Useful when sessions do not come with pre-written summaries.
        session_id: str, messages: list[dict] (N,) -> str (summary)
        '''

        text = ' '.join(
            m.get('content','') or '' for m in messages
            if m.get('role') in ('user','assistant') and m.get('content')
        )[:4_000]

        resp = client.chat.completions.create(
            model=MODEL,
            max_completion_tokens=80,
            messages=[
                {'role':'system','content':'Summarise this agent conversation in one sentence (max 20 words).'},
                {'role':'user','content': text},
            ],
        )

        summary = resp.choices[0].message.content.strip()
        self.index_session(session_id, messages, summary=summary)

        return summary

    def close(self):
        self._conn.close()


# === Demo ===
idx = SessionSearchIndex()

# Seed with past sessions
idx.index_session('sess_001',
    [{'role':'user','content':'Refactor the JWT auth module.'},
     {'role':'assistant','content':'Done. validate_token now uses RS256 and checks exp claim.'}],
    summary='Refactored JWT auth to RS256',
)
idx.index_session('sess_002',
    [{'role':'user','content':'Add rate limiting to the API gateway.'},
     {'role':'assistant','content':'Added token bucket rate limiter: 100 req/min per user.'}],
    summary='Added token bucket rate limiting to gateway',
)
idx.index_session('sess_003',
    [{'role':'user','content':'Fix the database connection pool leak.'},
     {'role':'assistant','content':'Found pool leak in db.py:88. Added explicit conn.close() in finally block.'}],
    summary='Fixed connection pool leak in db.py',
)

print('Cross-session search results for "auth JWT":')
for r in idx.search('auth JWT'):
    print(f'[{r["session_id"]}] {r["summary"]}')
    print(f'snippet: {r["snippet"]}')
    print(f'rank:    {r["rank"]}')

print('\nSearch results for "rate limit":')
for r in idx.search('rate limit'):
    print(f'[{r["session_id"]}] {r["summary"]}')

idx.close()

Cross-session search results for "auth JWT":
[sess_001] Refactored JWT auth to RS256
snippet: Refactor the [JWT] [auth] module. Done. validate_token now uses RS256 and checks exp claim.
rank:    -1.469

Search results for "rate limit":


# 2) Cost and Token Budgeting


LLM APIs charge per token. A runaway agent (infinite loop, accidentally reading a 100MB file, spawning too many subagents) can burn $100 in minutes.

Token budgeting adds two controls:
1. **Per-session budget**: a session cannot exceed N total tokens.
2. **Per-turn budget**: a single LLM call cannot exceed M output tokens.

Both are enforced before the API call, not after. By the time you see
the bill it is too late.

```python
budget = TokenBudget(session_limit=50_000, turn_limit=2_000)
budget.consume(input_tokens=500, output_tokens=200)
budget.remaining()   # -> 49_300
budget.check()       # -> raises BudgetExceeded if over limit
```

In [38]:
class BudgetExceeded(Exception):
    pass

# Approximate pricing per 1M tokens (USD) -- April 2026
COST_TABLE = {
    'gpt-4o-mini': {'input': 2.50,  'output': 10.00, 'cache': 1.25},
    'gpt-4o': {'input': 10.00, 'output': 30.00, 'cache': 1.50},
}


@dataclass
class TokenBudget:
    # Tracks token usage and enforces per-session and per-turn limits.
    session_limit: int = 100_000   # max total tokens for this session
    turn_limit:    int = 4_096     # max output tokens per turn
    model:         str = 'gpt-4o-mini'

    # Running totals
    total_input:   int = 0
    total_output:  int = 0
    total_cached:  int = 0
    turns:         int = 0

    @property
    def total_tokens(self):
        return self.total_input + self.total_output

    @property
    def remaining(self):
        return max(0, self.session_limit - self.total_tokens)

    def consume(self, input_tokens, output_tokens, cached_tokens=0):
        '''
        Record tokens from one LLM call.
        input_tokens: int, output_tokens: int -> None | raises BudgetExceeded
        '''

        self.total_input  += input_tokens - cached_tokens
        self.total_output += output_tokens
        self.total_cached += cached_tokens
        self.turns        += 1
        if self.total_tokens > self.session_limit:
            raise BudgetExceeded(
                f'Session budget exceeded: {self.total_tokens} > {self.session_limit} tokens'
            )

    def check_turn_limit(self, requested_max_tokens):
        '''
        Clamp requested_max_tokens to what the budget allows.
        requested_max_tokens: int -> int (safe max_tokens to pass to the API)
        '''
        available = min(self.remaining, self.turn_limit)
        return min(requested_max_tokens, available)

    def cost_usd(self):
        # -> float (estimated USD spent so far)
        table = COST_TABLE.get(self.model, COST_TABLE['gpt-4o-mini'])
        cost  = (self.total_input  / 1_000_000) * table['input']
        cost += (self.total_output / 1_000_000) * table['output']
        cost += (self.total_cached / 1_000_000) * table['cache']
        return round(cost, 6)

    def report(self):
        return (
            f'Budget Report [{self.model}]\n'
            f'  Turns:          {self.turns}\n'
            f'  Input tokens:   {self.total_input:,}\n'
            f'  Output tokens:  {self.total_output:,}\n'
            f'  Cached tokens:  {self.total_cached:,}\n'
            f'  Total tokens:   {self.total_tokens:,} / {self.session_limit:,}\n'
            f'  Remaining:      {self.remaining:,}\n'
            f'  Est. cost:      ${self.cost_usd():.4f} USD'
        )


def make_budget_hook(budget, logger=None):
    '''
    Factory: returns an on_response hook that tracks usage and enforces budget.
    budget: TokenBudget -> Callable(response) -> None
    '''

    def hook(response):
        usage = response.usage
        cached = getattr(usage, 'prompt_tokens_details', None)
        cached_tokens = getattr(cached, 'cached_tokens', 0) if cached else 0
        try:
            budget.consume(
                input_tokens = usage.prompt_tokens,
                output_tokens = usage.completion_tokens,
                cached_tokens = cached_tokens,
            )
        except BudgetExceeded as exc:
            if logger: logger.error('budget', str(exc))
            raise
    return hook


# === Demo ===
budget = TokenBudget(session_limit=10_000, turn_limit=500, model='gpt-4o')

# Simulate consuming tokens across 3 turns
budget.consume(input_tokens=300, output_tokens=150)
budget.consume(input_tokens=450, output_tokens=200, cached_tokens=200)
budget.consume(input_tokens=600, output_tokens=180)

print(budget.report())
print(f'\nTurn limit (requesting 1000): safe max = {budget.check_turn_limit(1000)}')

Budget Report [gpt-4o]
  Turns:          3
  Input tokens:   1,150
  Output tokens:  530
  Cached tokens:  200
  Total tokens:   1,680 / 10,000
  Remaining:      8,320
  Est. cost:      $0.0277 USD

Turn limit (requesting 1000): safe max = 500


# 3) Rate Limiting

Without rate limits, one user can exhaust your entire API quota, leaving
all other users unable to get a response. Rate limiting ensures fair
resource allocation across users and prevents abuse.

```
Level 1: Per-user limit    -- 10 requests / minute per peer_id
Level 2: Global limit      -- 500 requests / minute total
```

Both use the **token bucket** algorithm: each bucket holds up to `capacity`
tokens; tokens refill at `rate` per second; each request consumes 1 token.
If the bucket is empty, the request is rejected (or queued).

Token bucket allows short bursts (empty the full bucket instantly) then
enforces the average rate. Sliding window counts exact requests in the
last N seconds -- stricter, more memory-intensive.
Token bucket is the right default for API rate limiting.

In [39]:
class TokenBucket:
    def __init__(self, capacity, rate_per_second):
        # capacity: int -- max burst size
        # rate_per_second: float -- sustained request rate
        self._capacity = capacity
        self._rate     = rate_per_second
        self._tokens   = float(capacity)  # start full
        self._last     = time.monotonic()
        self._lock     = threading.Lock()

    def _refill(self):
        now    = time.monotonic()
        delta  = now - self._last
        self._tokens = min(self._capacity, self._tokens + delta * self._rate)
        self._last   = now

    def consume(self, tokens=1):
        # Try to consume `tokens` from the bucket.
        with self._lock:
            self._refill()
            if self._tokens >= tokens:
                self._tokens -= tokens
                return True
            return False

    @property
    def available(self):
        with self._lock:
            self._refill()
            return round(self._tokens, 2)


class RateLimiter:
    '''
    Two-level rate limiter: per-user and global.
    Uses token buckets for both levels.
    '''

    def __init__(self, per_user_rpm=10, global_rpm=500):

        self._per_user_rate = per_user_rpm / 60.0
        self._global_rate   = global_rpm   / 60.0
        # _user_buckets: dict[str, TokenBucket] (num_active_users,)
        self._user_buckets: dict = {}
        self._global_bucket  = TokenBucket(
            capacity=global_rpm, rate_per_second=self._global_rate
        )
        self._lock = threading.Lock()

        # _denied: dict[str, int] (num_denied_per_user)
        self._denied: dict = defaultdict(int)

    def _get_user_bucket(self, peer_id):
        with self._lock:
            if peer_id not in self._user_buckets:
                self._user_buckets[peer_id] = TokenBucket(
                    capacity=int(self._per_user_rate * 60),
                    rate_per_second=self._per_user_rate,
                )
            return self._user_buckets[peer_id]

    def allow(self, peer_id):
        # Check both levels. Returns True if the request is allowed.
        user_ok   = self._get_user_bucket(peer_id).consume()
        global_ok = self._global_bucket.consume() if user_ok else False
        allowed   = user_ok and global_ok
        if not allowed:
            self._denied[peer_id] += 1
        return allowed

    def stats(self):
        return {
            'global_available': self._global_bucket.available,
            'active_users': len(self._user_buckets),
            'total_denied': sum(self._denied.values()),
            'top_denied': sorted(self._denied.items(), key=lambda x: -x[1])[:5],
        }


# === Demo ===
limiter = RateLimiter(per_user_rpm=3, global_rpm=10)

results = []
for i in range(5):
    r = limiter.allow('user_a')
    results.append(('user_a', r))
    time.sleep(0.05)

for i in range(3):
    r = limiter.allow('user_b')
    results.append(('user_b', r))

print('Rate limit test (3 req/min per user):')
for peer, allowed in results:
    symbol = 'OK  ' if allowed else 'DENY'
    print(f'  [{symbol}] {peer}')

print(f'\nStats: {limiter.stats()}')

Rate limit test (3 req/min per user):
  [OK  ] user_a
  [OK  ] user_a
  [OK  ] user_a
  [DENY] user_a
  [DENY] user_a
  [OK  ] user_b
  [OK  ] user_b
  [OK  ] user_b

Stats: {'global_available': 4.04, 'active_users': 2, 'total_denied': 2, 'top_denied': [('user_a', 2)]}


# 4) Prompt Caching

OpenAI caches prompt prefixes automatically. If your system prompt is 2_000 tokens and you make 100 calls, you pay for 2_000 tokens once (full price) and 99 times at the cache rate (~50% discount).

**This only works if the prefix is identical across calls.**

Any change to the start of the prompt busts the cache. The golden rule:

```
STABLE (cache-friendly)    DYNAMIC (cache-busting)
  BASE prompt                current timestamp
  IDENTITY.md                MEMORY.md (changes per turn)
  SOUL.md                    CONTEXT (changes per turn)
  TOOLS.md
  HEARTBEAT.md
```

**Cache-Friendly Prompt Structure**

```
[STABLE PREFIX --- never changes]
  Layer 1: BASE
  Layer 2: IDENTITY
  Layer 3: SOUL
  Layer 4: TOOLS

[DYNAMIC SUFFIX --- changes every turn]
  Layer 5: MEMORY  (user-specific, changes rarely)
  Layer 6: SKILLS  (session-specific)
  Layer 7: CONTEXT (changes every turn: time, channel, peer)
  Layer 8: HEARTBEAT (only in heartbeat mode)
```

The stable prefix should be as long as possible to maximise cache hits.
Put the most stable content first, the most volatile content last.

OpenAI returns `usage.prompt_tokens_details.cached_tokens` on each
response. Tracking this tells you how well your prompt structure is
working and how much money you are saving.

In [40]:
import tiktoken

@dataclass
class CacheStats:
    # Tracks cache performance across an agent session.
    total_prompt_tokens: int = 0
    total_cached_tokens: int = 0
    calls: int = 0

    def record(self, prompt_tokens, cached_tokens):
        self.total_prompt_tokens += prompt_tokens
        self.total_cached_tokens += cached_tokens
        self.calls += 1

    @property
    def hit_rate(self):
        if self.total_prompt_tokens == 0:
            return 0.0
        return round(self.total_cached_tokens / self.total_prompt_tokens, 3)

    def savings_usd(self, model='gpt-4o-mini'):

        # Cost saved vs paying full price for cached tokens.
        table = COST_TABLE.get(model, COST_TABLE['gpt-4o-mini'])
        full_rate  = table['input']
        cache_rate = table['cache']
        saved_per_m = full_rate - cache_rate
        return round((self.total_cached_tokens / 1_000_000) * saved_per_m, 6)

    def report(self, model='gpt-4o'):
        return (
            f'Cache Stats:\n'
            f'  API calls:           {self.calls}\n'
            f'  Total prompt tokens: {self.total_prompt_tokens:,}\n'
            f'  Cached tokens:       {self.total_cached_tokens:,}\n'
            f'  Cache hit rate:      {self.hit_rate:.1%}\n'
            f'  Savings:             ${self.savings_usd(model):.4f} USD'
        )


class CacheAwarePromptBuilder:
    '''
    Builds prompts with stable prefix and dynamic suffix separated.
    The stable prefix should be maximally long for best cache performance.
    '''

    def __init__(self, stable_layers, model='gpt-4o'):
        '''
        stable_layers: list[str] -- content that never changes across calls
        Joined once at construction; never rebuilt.
        stable_layers: list[str] (num_stable,) -> str (cached prefix)
        '''

        self._stable = '\n\n'.join(stable_layers)
        self._encoder = tiktoken.encoding_for_model(model)
        self._stable_tokens = len(self._encoder.encode(self._stable))

    def build(self, dynamic_layers):
        # dynamic_layers: list[str] -- content that changes every call
        dynamic = '\n\n'.join(dynamic_layers)

        # (full prompt = stable prefix + dynamic suffix)
        return self._stable + ('\n\n' + dynamic if dynamic else '')

    @property
    def stable_preview(self):
        return self._stable[:200] + ('...' if len(self._stable) > 200 else '')


def make_cache_hook(stats):
    # Factory: returns an on_response hook that records cache hit data.
    # stats: CacheStats -> Callable(response) -> None
    def hook(response):
        usage  = response.usage
        cached = getattr(
            getattr(usage, 'prompt_tokens_details', None),
            'cached_tokens', 0) or 0

        stats.record(prompt_tokens=usage.prompt_tokens, cached_tokens=cached)
    return hook


# === Demo ===
stable_layers = [
    '# Role\nYou are a helpful AI assistant with tool use capabilities.',
    '# Identity\nYou are Claw, deployed in a messaging gateway.',
    '# Soul\nBe concise, proactive, and honest. Avoid filler phrases.',
    '# Tools\nbash: run shell commands | file_read: read files | file_write: write files',
]

dynamic_layers = [
    f'# Context\nTime: {datetime.now().isoformat()}\nChannel: telegram\nPeer: 12345',
    '# Memory\nUser prefers bullet points.',
]

builder = CacheAwarePromptBuilder(stable_layers)
full_prompt = builder.build(dynamic_layers)

enc = tiktoken.encoding_for_model('gpt-4o-mini')
full_tokens = len(enc.encode(full_prompt))

print(f'Stable prefix: {builder._stable_tokens} tokens')
print(f'Full prompt:   {full_tokens} tokens')
print(f'Cache-able:    {builder._stable_tokens / full_tokens:.1%} of prompt')
print()

cache_stats = CacheStats()
# Simulate 10 calls with 70% cache hit rate
for i in range(10):
    cached = int(800 * 0.7) if i > 0 else 0  # first call has no cache
    cache_stats.record(prompt_tokens=1_000, cached_tokens=cached)
print(cache_stats.report())

Stable prefix: 63 tokens
Full prompt:   104 tokens
Cache-able:    60.6% of prompt

Cache Stats:
  API calls:           10
  Total prompt tokens: 10,000
  Cached tokens:       5,040
  Cache hit rate:      50.4%
  Savings:             $0.0428 USD


# 5) Multi-Model Routing

Not all tasks need a frontier model. A routing layer that sends simple
requests to a cheaper model and complex ones to a more capable model
can cut costs by 70-90% without a noticeable quality drop.


| Signal | Simple task | Complex task |
|--------|-------------|--------------|
| Message length | < 50 words | > 200 words |
| Tool calls expected | 0 | 3+ |
| Keyword heuristics | 'what time', 'hello' | 'refactor', 'debug', 'analyse' |
| Explicit user flag | `--fast` | `--deep` |
| Confidence threshold | classifier > 0.9 | classifier < 0.5 |

The simplest effective strategy is a **keyword + length heuristic**.
A more sophisticated approach calls a tiny classifier model (gpt-4o-mini)
to score complexity before routing.


```python
router = ModelRouter(fast='gpt-4o-mini', smart='gpt-4o')
router.route('What time is it?')
router.route('Refactor my auth module to use OAuth2.')
```

In [41]:
SYSTEM_ROUTE_PROMPT = """
You are a model routing assistant. Determine if the user request is 'simple' or 'complex'. \
Simple tasks (greetings, basic facts, short queries) use 'fast'. \
Complex tasks (refactoring, debugging, deep analysis) use 'smart'. \
Respond ONLY in JSON format like: {\"route\": \"fast\" or \"smart\", \"reason\": \"<brief reason>\"}
"""

class RoutingDecision:
    def __init__(self, model, reason, confidence):
        self.model = model
        self.reason = reason
        self.confidence = confidence  # 0.0 to 1.0

    def __repr__(self):
        return f"RoutingDecision(model={self.model!r}, reason={self.reason!r}, conf={self.confidence:.2f})"


class ModelRouter:
    """
    Routes requests to fast or smart models based on LLM scoring.
    """

    def __init__(self, fast="gpt-4o-mini", smart="gpt-4o"):
        self.fast = fast
        self.smart = smart
        self._log: list = []

    def route(self, text, expected_tools=0, force=None):
        # text: str, expected_tools: int, force: str|None -> RoutingDecision
        if force:
            return RoutingDecision(force, "forced", 1.0)

        try:
            resp = client.chat.completions.create(
                model=self.fast,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_ROUTE_PROMPT,
                    },
                    {"role": "user", "content": text[:1000]},
                ],
                response_format={"type": "json_object"},
                max_completion_tokens=50,
            )
            content = resp.choices[0].message.content
            if not content:
                content = '{"route": "fast", "reason": "empty_response"}'

            result = json.loads(content)
            route_choice = result.get("route", "fast").lower()
            reason = result.get("reason", "llm_evaluated")

            if route_choice == "smart":
                d = RoutingDecision(self.smart, reason, 0.90)
            else:
                d = RoutingDecision(self.fast, reason, 0.90)

        except Exception as e:
            d = RoutingDecision(self.fast, f"fallback_due_to_error: {e}", 0.50)

        self._log.append({"text": text[:60], "model": d.model, "reason": d.reason})
        return d

    def routing_stats(self):
        # -> dict (fast_count, smart_count, fast_pct)
        fast = sum(1 for r in self._log if r["model"] == self.fast)
        smart = sum(1 for r in self._log if r["model"] == self.smart)
        total = len(self._log)
        return {
            "fast": fast,
            "smart": smart,
            "total": total,
            "fast_pct": f"{fast/total:.0%}" if total else "N/A",
        }


# === Demo ===
router = ModelRouter(fast="gpt-4o-mini", smart="gpt-4o")

test_cases = [
    "Hello there!",
    "What time is it?",
    "Refactor the authentication module to use OAuth2 with PKCE.",
    "Why does my Redis connection keep dropping under load?",
    "List the top 3 Python web frameworks.",
    "Analyse the performance bottleneck in my database query pipeline "
    "and suggest an indexing strategy to reduce p99 latency below 50ms.",
]

print("Routing decisions:")
for text in test_cases:
    d = router.route(text)
    print(f"  [{d.model:15}] {text[:55]}... -> {d.reason}")

print(f"\nStats: {router.routing_stats()}")


Routing decisions:
  [gpt-4o-mini    ] Hello there!... -> Greeting and simple interaction.
  [gpt-4o-mini    ] What time is it?... -> This is a straightforward query requesting a basic fact.
  [gpt-4o         ] Refactor the authentication module to use OAuth2 with P... -> The task involves refactoring code and requires a deep understanding of OAuth2 with PKCE.
  [gpt-4o         ] Why does my Redis connection keep dropping under load?... -> The request requires troubleshooting and analyzing connection issues under load conditions.
  [gpt-4o-mini    ] List the top 3 Python web frameworks.... -> The request is a basic fact query.
  [gpt-4o         ] Analyse the performance bottleneck in my database query... -> The request involves analyzing performance issues and suggesting an indexing strategy, which requires deep analysis and understanding of database optimization.

Stats: {'fast': 3, 'smart': 3, 'total': 6, 'fast_pct': '50%'}


# 6) Health Checks and Circuit Breakers

When an external API (OpenAI, Telegram, Feishu) is degraded, every request
fails and waits for the full timeout before giving up. Under load this
cascades: threads pile up, memory fills, the entire gateway becomes unresponsive.

The **circuit breaker** pattern prevents this. It works like an electrical
circuit breaker: when too many failures occur, it trips OPEN and rejects
all requests immediately (no waiting). After a cooldown it goes HALF-OPEN
to probe recovery. On success it closes.

```
CLOSED (normal) -----failures exceed threshold-----> OPEN (rejecting)
                                                         |
   ^                                                 cooldown elapsed
   |                                                     |
   \--- probe succeeds --- HALF-OPEN (probing) <---------/
```

A health check runs on a schedule and measures API availability.
The circuit breaker uses health check results to update its state.

In [46]:
class CircuitState(Enum):
    CLOSED    = auto()  # normal -- requests pass through
    OPEN      = auto()  # tripped -- all requests rejected immediately
    HALF_OPEN = auto()  # probing -- one request allowed to test recovery


class CircuitBreaker:
    # Protects a downstream service from cascade failures.
    # Thread-safe.

    def __init__(self, failure_threshold=5, cooldown_s=30, probe_timeout_s=5):
        # failure_threshold: int -- consecutive failures to trip OPEN
        # cooldown_s: float -- time in OPEN before moving to HALF_OPEN
        self._threshold      = failure_threshold
        self._cooldown       = cooldown_s
        self._probe_timeout  = probe_timeout_s
        self._state          = CircuitState.CLOSED
        self._failure_count  = 0
        self._opened_at:  Optional[float] = None
        self._lock = threading.Lock()

    @property
    def state(self): return self._state

    def allow(self):
        # -> bool (True if the request should be attempted)
        with self._lock:
            if self._state == CircuitState.CLOSED:
                return True
            if self._state == CircuitState.OPEN:
                if time.monotonic() - self._opened_at >= self._cooldown:
                    self._state = CircuitState.HALF_OPEN
                    print(f'  [Circuit] -> HALF_OPEN (probing after {self._cooldown}s cooldown)')
                    return True  # allow probe
                return False  # still cooling down
            # HALF_OPEN: allow exactly one probe
            return True

    def record_success(self):
        with self._lock:
            if self._state == CircuitState.HALF_OPEN:
                print('  [Circuit] Probe succeeded -> CLOSED')
            self._state         = CircuitState.CLOSED
            self._failure_count = 0
            self._opened_at     = None

    def record_failure(self):
        with self._lock:
            self._failure_count += 1
            if self._state == CircuitState.HALF_OPEN:
                # Probe failed -- back to OPEN
                self._state     = CircuitState.OPEN
                self._opened_at = time.monotonic()
                print('  [Circuit] Probe failed -> OPEN')
            elif self._failure_count >= self._threshold:
                self._state     = CircuitState.OPEN
                self._opened_at = time.monotonic()
                print(f'  [Circuit] {self._failure_count} failures -> OPEN')

    def call(self, fn, *args, fallback=None, **kwargs):
        # Attempt fn(*args, **kwargs) with circuit breaker protection.
        # fn: Callable -> result | fallback
        if not self.allow():
            if fallback is not None:
                return fallback
            raise RuntimeError(f'Circuit is OPEN. Service unavailable.')
        try:
            result = fn(*args, **kwargs)
            self.record_success()
            return result
        except Exception as exc:
            self.record_failure()
            if fallback is not None: return fallback
            raise


def run_health_check(cb, check_fn, interval_s=10):
    # Run check_fn on a schedule; report results to circuit breaker.
    # check_fn: Callable() -> bool (True = healthy)
    # interval_s: float -> threading.Thread
    stop = threading.Event()
    def _loop():
        while not stop.is_set():
            try:
                ok = check_fn()
                if ok: cb.record_success()
                else:  cb.record_failure()
            except Exception:
                cb.record_failure()
            stop.wait(timeout=interval_s)
    t = threading.Thread(target=_loop, daemon=True)
    t.start()
    return t, stop


# === Demo ===
cb = CircuitBreaker(failure_threshold=3, cooldown_s=2)

def unstable_service(should_fail=False):
    if should_fail: raise ConnectionError('Service unavailable')
    return 'OK'

print('Triggering failures to trip circuit:')
for i in range(4):
    try:
        result = cb.call(unstable_service, should_fail=True, fallback='(fallback)')
        print(f'  Call {i+1}: {result} [{cb.state.name}]')
    except Exception as exc:
        print(f'  Call {i+1}: Error [{cb.state.name}]')

print(f'\nAfter failures: {cb.state.name}')
time.sleep(2.1)  # let cooldown expire
print(f'After cooldown: {cb.state.name} (should be HALF_OPEN or reset)')
result = cb.call(unstable_service, should_fail=False)
print(f'Probe result: {result} [{cb.state.name}]')


Triggering failures to trip circuit:
  Call 1: (fallback) [CLOSED]
  Call 2: (fallback) [CLOSED]
  [Circuit] 3 failures -> OPEN
  Call 3: (fallback) [OPEN]
  Call 4: (fallback) [OPEN]

After failures: OPEN
After cooldown: OPEN (should be HALF_OPEN or reset)
  [Circuit] -> HALF_OPEN (probing after 2s cooldown)
  [Circuit] Probe succeeded -> CLOSED
Probe result: OK [CLOSED]


# 7) Agent Testing

Testing agents is hard because they are non-deterministic: the same
prompt can produce different tool call sequences on different runs.
Three strategies that work:

1. **Mock LLM testing**: replace the LLM with a deterministic mock.
   Test that your harness correctly handles tool calls, errors, compaction.
   Fast (no API calls), reliable, cheap.

2. **Golden output testing**: record one real LLM run, save it as a
   golden fixture. Replay it to verify your harness produces the same
   final output given the same messages.

3. **Behavioural assertions**: run against the real LLM and assert on
   *properties* (response length, tool was called, no error in output)
   rather than exact text.

```python
mock = MockLLM(responses=[
    LLMResponse(tool_calls=[('bash', {'command': 'ls -la'})]),
    LLMResponse(text='Found 3 files.'),
])
with mock.patch():     # temporarily replaces client.chat.completions.create
    run_agent_loop(...) # harness calls mock instead of real API
mock.assert_called_with_tools(['bash'])
```

In [43]:
from unittest.mock import MagicMock, patch

@dataclass
class MockLLMResponse:
    # Represents one scripted LLM response for testing.
    text:       Optional[str] = None
    tool_calls: list = field(default_factory=list)
    # tool_calls: list[(name: str, args: dict)]


class MockLLM:
    '''
    Deterministic LLM mock for unit testing agent harness code.
    Replaces client.chat.completions.create with scripted responses.
    Does not make any API calls.
    '''

    def __init__(self, responses):
        # responses: list[MockLLMResponse] -- consumed in order
        self._queue    = deque(responses)
        self._calls: list = []  # all calls recorded here

    def _make_response(self, scripted):
        # Build an object that looks like a real OpenAI ChatCompletion.
        choice = MagicMock()
        choice.message.content = scripted.text
        choice.message.tool_calls = None
        choice.finish_reason = 'stop'

        if scripted.tool_calls:
            choice.finish_reason = 'tool_calls'
            tcs = []

            for name, args in scripted.tool_calls:
                tc = MagicMock()
                tc.id = uuid.uuid4().hex[:8]
                tc.function.name  = name
                tc.function.arguments = json.dumps(args)
                tcs.append(tc)
            choice.message.tool_calls = tcs

        resp = MagicMock()
        resp.choices = [choice]
        resp.usage.prompt_tokens = 100
        resp.usage.completion_tokens = 50
        return resp

    def create(self, **kwargs):
        # Drop-in replacement for client.chat.completions.create
        self._calls.append(kwargs)
        if not self._queue:
            # Default: empty stop response when queue is exhausted
            scripted = MockLLMResponse(text='(no more scripted responses)')
        else:
            scripted = self._queue.popleft()
        return self._make_response(scripted)

    def assert_tool_called(self, tool_name):
        called = []
        for call in self._calls:
            msgs = call.get('messages', [])
            for m in msgs:
                if isinstance(m, dict) and m.get('role') == 'tool':
                    called.append(m.get('content', ''))
        # Check that any assistant message requested this tool
        for call in self._calls:
            for m in call.get('messages', []):
                if isinstance(m, dict):
                    for tc in m.get('tool_calls', []) or []:
                        if isinstance(tc, dict) and tc.get('function', {}).get('name') == tool_name:
                            return True
        raise AssertionError(f'Tool {tool_name!r} was never called.')

    @property
    def call_count(self): return len(self._calls)


class GoldenTest:
    '''
    Record a real agent run; replay it to verify correctness.
    The golden fixture captures the full messages[] list.
    '''

    def __init__(self, fixture_path):
        self._path = Path(fixture_path)

    def record(self, messages):
        # Save messages as the golden fixture.
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._path.write_text(json.dumps(messages, indent=2, default=str))
        print(f'Recorded {len(messages)} messages to {self._path}')

    def load(self):
        # Load the golden fixture.
        if not self._path.exists():
            raise FileNotFoundError(f'Golden fixture not found: {self._path}')
        return json.loads(self._path.read_text())

    def assert_final_text_contains(self, messages, substring):
        # Assert the last assistant message contains a substring.
        last = next(
            (m['content'] for m in reversed(messages)
             if m.get('role') == 'assistant' and m.get('content')),
            None,
        )
        if last is None:
            raise AssertionError('No assistant message found.')
        if substring.lower() not in last.lower():
            raise AssertionError(f'{substring!r} not found in: {last[:100]}')
        return True


# === Demo 1: Mock LLM test (Unit Testing Harness) ===
print('=== Demo 1: Mock LLM Test (Infrastructure) ===')
mock = MockLLM(responses=[
    MockLLMResponse(tool_calls=[('bash', {'command': 'echo hello'})]),
    MockLLMResponse(text='The command output was: hello'),
])

# Patch the real client with our mock
# Intercept client wotk MockLLM, and reply `Call bash tool!`
# Test whether the loop able to find the bash tool and run the function
with patch.object(client.chat.completions, 'create', side_effect=mock.create):
    messages = [{'role':'user','content':'Run echo hello and tell me the output.'}]
    run_agent_loop(
        messages=messages,
        tools=[{'type':'function','function':{'name':'bash','description':'Run shell',
                'parameters':{'type':'object','properties':{'command':{'type':'string'}},
                              'required':['command']}}}],
        tool_handlers={'bash': lambda command: 'hello'},
        system='You are a test assistant.',
    )

print(f'Mock LLM test passed!')
print(f'  LLM calls made: {mock.call_count}')
print(f'  Final reply: {messages[-1]["content"]}')

# Verify the bash tool was called
try:
    mock.assert_tool_called('bash')
    print('  assert_tool_called(bash): PASS')
except AssertionError as e:
    print(f'  FAIL: {e}')


# === Demo 2: Real LLM test (Integration Testing Intelligence) ===
print('\n=== Demo 2: Real LLM Test (Intelligence) ===')
real_messages = [{'role': 'user', 'content': 'Calculate 15 * 7 and reply with just the number.'}]

# This runs against the REAL API, proving end-to-end capabilities
run_agent_loop(
    messages=real_messages,
    tools=[],
    tool_handlers={},
    system='You are a helpful assistant.',
)
print(f'Real LLM test passed!')
print(f'  Final reply: {real_messages[-1].get("content")}')


=== Demo 1: Mock LLM Test (Infrastructure) ===
Mock LLM test passed!
  LLM calls made: 2
  Final reply: The command output was: hello
  assert_tool_called(bash): PASS

=== Demo 2: Real LLM Test (Intelligence) ===
Real LLM test passed!
  Final reply: 105


# 8) Configuration Management

Agents have many configuration knobs: which model to use, session token
budgets, rate limits, heartbeat intervals, delivery retry counts.
Hard-coding these creates brittle code that requires a redeploy for every
tuning change.

**Configuration management** separates settings from code:
- `.env` files for secrets (API keys)
- `config.json` or environment variables for operational settings
- **Feature flags** for gradual rollouts (enable multi-model routing for
  10% of users before full rollout)

## 8.2 Layered Config Precedence

```
1. Defaults (lowest priority)  -- hardcoded in code
2. config.json                 -- version-controlled defaults
3. Environment variables       -- deployment-specific overrides
4. Runtime overrides           -- in-flight changes (feature flags)
```

Higher layers always win. This lets you set sensible defaults in code,
override them per-environment via env vars, and A/B test via flags.

In [44]:
@dataclass
class AgentConfig:
    '''
    All configuration knobs for a production agent deployment.
    Populated from defaults, then config file, then env vars.
    '''

    # Model settings
    model:              str   = 'gpt-4o'
    fast_model:         str   = 'gpt-4o-mini'
    max_output_tokens:  int   = 4_096

    # Budget
    session_token_limit: int  = 100_000
    turn_token_limit:    int  = 4_096

    # Rate limiting
    per_user_rpm:       int   = 20
    global_rpm:         int   = 500

    # Resilience
    tool_retry_max:     int   = 3
    auth_retry_max:     int   = 3
    circuit_threshold:  int   = 5
    circuit_cooldown_s: int   = 30

    # Heartbeat
    heartbeat_interval_s: int = 300

    # Delivery
    delivery_max_attempts: int = 5

    # Feature flags
    enable_multi_model:   bool = False
    enable_caching:       bool = True
    enable_heartbeat:     bool = True

    @classmethod
    def from_env(cls, config_path=None):
        '''
        Load from defaults, then config file, then environment variables.
        config_path: str|None -> AgentConfig
        '''
        cfg = cls()  # start with defaults

        # Layer 2: config file
        if config_path and Path(config_path).exists():
            data = json.loads(Path(config_path).read_text())
            for k, v in data.items():
                if hasattr(cfg, k): setattr(cfg, k, v)

        # Layer 3: environment variables (AGENT_ prefix)
        for f in cfg.__dataclass_fields__:
            env_key = f'AGENT_{f.upper()}'
            val     = os.environ.get(env_key)
            if val is not None:
                field_type = type(getattr(cfg, f))
                if field_type == bool:
                    setattr(cfg, f, val.lower() in ('1','true','yes'))
                else:
                    try: setattr(cfg, f, field_type(val))
                    except (ValueError, TypeError): pass
        return cfg

    def to_dict(self): return dict(self.__dict__)


class FeatureFlagStore:
    '''
    Simple in-memory feature flag store with per-user rollout support.
    In production, back this with LaunchDarkly, Statsig, or a DB.
    '''

    def __init__(self):
        self._flags: dict = {}
        self._lock = threading.Lock()

    def define(self, name, enabled=False, rollout_pct=0, allowlist=None):
        with self._lock:
            self._flags[name] = {
                'enabled':      enabled,
                'rollout_pct':  rollout_pct,
                'allowlist':    set(allowlist or []),
            }

    def is_enabled(self, name, peer_id=None):
        # Check if a feature is enabled for a specific user.
        with self._lock:

            flag = self._flags.get(name)
            if not flag:
                return False

            if not flag['enabled']:
                return False
            if peer_id and peer_id in flag['allowlist']:
                return True

            if flag['rollout_pct'] >= 100:
                return True

            if peer_id and flag['rollout_pct'] > 0:
                # Deterministic hash-based rollout: same user always gets same result
                bucket = int(uuid.uuid5(uuid.NAMESPACE_DNS, peer_id).int % 100)
                return bucket < flag['rollout_pct']

            return False


# === Demo ===
cfg = AgentConfig.from_env()
print('Default config:')
for k, v in list(cfg.to_dict().items())[:8]:
    print(f'  {k}: {v}')
print('  ...')

flags = FeatureFlagStore()
flags.define('multi_model_routing', enabled=True, rollout_pct=50)
flags.define('heartbeat_v2', enabled=True, allowlist=['alice', 'bob'])

print('\nFeature flags:')
test_users = ['alice', 'charlie', 'delta', 'echo']
for user in test_users:
    mm = flags.is_enabled('multi_model_routing', peer_id=user)
    hb = flags.is_enabled('heartbeat_v2', peer_id=user)
    print(f'  {user:10}: multi_model={mm}, heartbeat_v2={hb}')

Default config:
  model: gpt-4o
  fast_model: gpt-4o-mini
  max_output_tokens: 4096
  session_token_limit: 100000
  turn_token_limit: 4096
  per_user_rpm: 20
  global_rpm: 500
  tool_retry_max: 3
  ...

Feature flags:
  alice     : multi_model=False, heartbeat_v2=True
  charlie   : multi_model=False, heartbeat_v2=False
  delta     : multi_model=True, heartbeat_v2=False
  echo      : multi_model=True, heartbeat_v2=False


# 9) Execution Environments

Your trilogy assumes the agent runs locally (subprocess.run talks to the
host machine). Hermes treats the execution backend as a **pluggable strategy**:

The agent code is identical; only the execution adapter changes.

| Backend | When to use | Isolation |
|---------|-------------|-----------|
| Local | Dev / laptop | None -- agent touches your real filesystem |
| Docker | Staging / CI | Container -- clean environment per session |
| SSH | Remote VM / VPS | Network -- agent runs on a $5 cloud server |
| Daytona | Cloud dev env | Serverless -- hibernates when idle, wakes on demand |
| Modal | GPU workloads | Serverless + GPU -- costs nothing between runs |
| Singularity | HPC clusters | Container -- no root required, runs on SLURM |

**The interface is identical from the agent loop perspective.**
All backends implement the same two methods:

```python
class ExecutionBackend:
    def run_command(self, command: str) -> str: ...
    def write_file(self, path: str, content: str) -> str: ...
```

The harness injects the backend at startup:

```python
backend = DockerBackend(image='python:3.11-slim')
# or:  SSHBackend(host='my-vps.com', user='ubuntu', key='~/.ssh/id_rsa')
# or:  ModalBackend(app_name='hermes', gpu='A100')

tool_handlers = {
    'bash':       lambda cmd: backend.run_command(cmd),
    'file_write': lambda path, content: backend.write_file(path, content),
}
# All other harness code is unchanged.
```

This is the **strategy pattern** applied to agent infrastructure.
The same agent can be used for a quick laptop demo (Local),
a reproducible research experiment (Docker), or a production deployment
that costs nearly nothing when idle (Daytona / Modal).
The separation between *what the agent does* and *where it runs* is
one of the cleanest architectural decisions in Hermes.

# 11) The Self-Improving Loop

Most agents are static: the model weights never change after deployment.
Hermes closes a second loop beyond skill creation:
it **records agent trajectories and feeds them to RL training**,
so the underlying model improves from the agent's own experience.

```
Agent runs a task
     |
     v
TrajectoryRecorder captures: prompt, tool_calls, results, final_reply
     |
     v
TrajectoryCompressor converts to training format (prompt/completion pairs)
     |
     v
Atropos RL environment scores the trajectory (outcome reward)
     |
     v
GRPO / PPO update improves the model on tool-use tasks
     |
     v
Next agent version is better at the tasks it has actually done
```

A trajectory is the full record of one agent task:

```json
{
  "trajectory_id": "traj_abc123",
  "task": "Find and fix the bug in auth.py",
  "turns": [
    {"role": "user",      "content": "Find and fix the bug in auth.py"},
    {"role": "assistant", "tool_calls": [{"name": "file_read", "args": {"file_path": "auth.py"}}]},
    {"role": "tool",      "content": "def validate_token(tok):\n  ..."},
    {"role": "assistant", "tool_calls": [{"name": "file_write", "args": {"file_path": "auth.py", "content": "...fixed..."}}]},
    {"role": "tool",      "content": "Wrote 847 bytes to auth.py"},
    {"role": "assistant", "content": "Fixed: added missing exp claim check in validate_token."}
  ],
  "outcome": {"tests_passed": true, "reward": 1.0}
}
```

| Property | Good | Bad |
|----------|------|-----|
| Outcome signal | Binary (tests pass / fail) or scalar | Subjective / human-rated only |
| Tool call quality | Minimal, targeted calls | Redundant or incorrect calls |
| Trajectory length | Under 20 turns | 50+ turns (too noisy) |
| Task diversity | Many different task types | All the same task repeated |


The only missing piece is the **outcome signal** (did the task succeed?)
and the **RL training loop** (Atropos / GRPO). Both are outside the scope
of a harness engineering course but are the natural next step for
students moving from systems to research.

**Implementation pointer:** see `trajectory_compressor.py` and `rl_cli.py`
in the Hermes repo, and the Atropos paper (NousResearch, 2025) for the
RL environment design.

# 11) Complete ProductionAgent

A single `run()` call that wires all three notebooks together:
configuration loaded from env, request rate-checked, model routed,
circuit breaker checked, agent loop run with observability hooks,
token budget tracked, reply delivered.

In [45]:
class ProductionAgent:
    '''
    Designed to be instantiated once per deployment.
    '''

    def __init__(self, config=None):
        self.cfg     = config or AgentConfig.from_env()
        self.flags   = FeatureFlagStore()

        # Ops layer
        self.logger  = AgentLogger()
        self.budget  = TokenBudget(
            session_limit = self.cfg.session_token_limit,
            turn_limit    = self.cfg.turn_token_limit,
            model         = self.cfg.model,
        )
        self.router  = ModelRouter(
            fast  = self.cfg.fast_model,
            smart = self.cfg.model,
        )
        self.limiter = RateLimiter(
            per_user_rpm = self.cfg.per_user_rpm,
            global_rpm   = self.cfg.global_rpm,
        )
        self.cb      = CircuitBreaker(
            failure_threshold = self.cfg.circuit_threshold,
            cooldown_s        = self.cfg.circuit_cooldown_s,
        )
        self.cache_stats = CacheStats()

        # Session store
        # messages: dict[str, list] -- per-peer conversation history
        self._sessions: dict = {}

    def _get_session(self, peer_id):
        if peer_id not in self._sessions:
            self._sessions[peer_id] = []
        return self._sessions[peer_id]

    def _select_model(self, text, peer_id):
        # Use multi-model routing if enabled for this user.
        if self.flags.is_enabled('multi_model_routing', peer_id=peer_id):
            decision = self.router.route(text)
            return decision.model
        return self.cfg.model

    def run(self, text, peer_id='anonymous', tools=None, tool_handlers=None):
        # Main entry point.
        # text: str, peer_id: str -> str (agent reply) | error message
        session_id = f'{peer_id}_{datetime.now().date()}'

        # Rate limiting
        if not self.limiter.allow(peer_id):
            self.logger.error(session_id, 'Rate limit exceeded')
            return f'[RateLimit] Too many requests. Please wait.'

        # Circuit breaker
        if not self.cb.allow():
            return f'[CircuitOpen] API unavailable. Please try again shortly.'

        # Model selection
        model = self._select_model(text, peer_id)

        # Observability setup
        tracer = Tracer(session_id=session_id)
        root   = tracer.start_span('user_turn', peer_id=peer_id, model=model)

        budget_hook = make_budget_hook(self.budget, self.logger)
        cache_hook  = make_cache_hook(self.cache_stats)

        def on_response(resp):
            budget_hook(resp)
            cache_hook(resp)
            usage = resp.usage
            self.logger.llm_response(
                session_id   = session_id,
                latency_ms   = 0,  # timing tracked by tracer
                output_tokens= usage.completion_tokens,
                finish_reason= resp.choices[0].finish_reason,
            )

        # Agent call (via circuit breaker)
        messages = list(self._get_session(peer_id))
        messages.append({'role':'user','content': text})

        try:
            self.cb.call(
                run_agent_loop,
                messages=messages,
                tools=tools or [],
                tool_handlers=tool_handlers or {},
                system=f'You are a helpful AI agent. Session: {session_id}.',
                on_response=on_response,
            )
            self.cb.record_success()
        except BudgetExceeded as exc:
            root.finish(error=str(exc))
            return f'[BudgetExceeded] {exc}'
        except Exception as exc:
            self.cb.record_failure()
            root.finish(error=str(exc))
            self.logger.error(session_id, str(exc), exc=exc)
            return f'[Error] {exc}'

        root.finish()

        # Update session
        self._sessions[peer_id] = messages

        reply = next(
            (m['content'] for m in reversed(messages)
             if m['role']=='assistant' and m['content']),
            '(no reply)',
        )
        return reply

    def ops_report(self):
        return (
            f'=== Ops Report ===\n'
            f'{self.budget.report()}\n\n'
            f'{self.cache_stats.report()}\n\n'
            f'{self.logger.metrics.report()}\n\n'
            f'Rate limiter: {self.limiter.stats()}\n'
            f'Circuit state: {self.cb.state.name}\n'
            f'Model routing: {self.router.routing_stats()}'
        )


# Capstone Demo
print('Initialising ProductionAgent...')
agent = ProductionAgent()
agent.flags.define('multi_model_routing', enabled=True, rollout_pct=100)

print('='*60)
print('Turn 1: Simple greeting')
print('='*60)
r1 = agent.run('Hello! What can you do?', peer_id='alice')
print(f'Reply: {r1[:200]}')

print('\n' + '='*60)
print('Turn 2: Complex question (multi-turn context retained)')
print('='*60)
r2 = agent.run('Explain the circuit breaker pattern in 2 sentences.', peer_id='alice')
print(f'Reply: {r2[:200]}')

print('\n' + '='*60)
print('Ops Report')
print('='*60)
print(agent.ops_report())

Initialising ProductionAgent...
Turn 1: Simple greeting
Reply: Hello! I can assist you with a variety of tasks, such as answering questions, providing information on various topics, helping with problem-solving, generating ideas, and much more. If you have someth

Turn 2: Complex question (multi-turn context retained)
Reply: The circuit breaker pattern is a design pattern used in software development to enhance the stability of a system by preventing it from repeatedly trying to execute an operation that is likely to fail

Ops Report
=== Ops Report ===
Budget Report [gpt-4o]
  Turns:          2
  Input tokens:   138
  Output tokens:  121
  Cached tokens:  0
  Total tokens:   259 / 100,000
  Remaining:      99,741
  Est. cost:      $0.0050 USD

Cache Stats:
  API calls:           2
  Total prompt tokens: 138
  Cached tokens:       0
  Cache hit rate:      0.0%
  Savings:             $0.0000 USD

Metrics:
  llm_latency_ms: {'count': 2, 'mean': 0, 'p50': 0, 'p95': 0, 'p99': 0, 'min': 0, '

# Summary: The Trilogy

```
+--------------------------------------------------------+
|                    Core                                |
|  The engine: how an agent thinks and acts              |
|                                                        |
|  run_agent_loop    TodoManager    FileTaskStore        |
|  SkillLibrary      compact_messages                    |
|  AutonomousWorker  WorktreeManager  AgentHarness       |
+--------------------------|-----------------------------+
                           |
+--------------------------|-----------------------------+
|                    Gateway                             |
|  The chassis: how it connects to the world             |
|                                                        |
|  PromptAssembler   InboundMessage  MessageGateway      |
|  HeartbeatScheduler CronScheduler  DeliveryQueue       |
|  ResilientRunner   LaneRouter     GatewayHarness       |
+--------------------------|-----------------------------+
                           |
+--------------------------|-----------------------------+
|                    Operation                           |
|  The factory: how you run it reliably at scale         |
|                                                        |
|  AgentLogger   TokenBudget   CacheStats                |
|  ModelRouter   RateLimiter   CircuitBreaker            |
|  MockLLM       GoldenTest    AgentConfig               |
|  FeatureFlagStore            ProductionAgent           |
+--------------------------------------------------------+
```

## Further Reading

- [shareAI-lab/learn-claude-code](https://github.com/shareAI-lab/learn-claude-code) -- agent core reference (12 sessions)
- [shareAI-lab/claw0](https://github.com/shareAI-lab/claw0) -- gateway reference (10 sessions)
- [OpenTelemetry Python](https://opentelemetry-python.readthedocs.io/) -- production tracing
- [openai-cookbook](https://github.com/openai/openai-cookbook) -- prompt caching patterns
- [Martin Fowler: Circuit Breaker](https://martinfowler.com/bliki/CircuitBreaker.html)